# Checando conexão do Microfone:

In [ ]:
!pwd

In [ ]:
!arecord -l

In [ ]:
!ls

In [ ]:
!sudo apt-get install portaudio19-dev python3-dev -y

In [ ]:
!pip install pyaudio

In [ ]:
import pyaudio

index = 0

p = pyaudio.PyAudio()
info = p.get_host_api_info_by_index(index)
numdevices = info.get('deviceCount')

print("Dispositivos de áudio encontrados:")
for i in range(0, numdevices):
    device_info = p.get_device_info_by_host_api_device_index(index, i)
    if device_info.get('maxInputChannels') > 0:
        print(f"  Índice de Entrada {i}: {device_info.get('name')}")

p.terminate()

In [ ]:
import pyaudio
import wave
import numpy as np
print(np.__version__)
from scipy.signal import resample_poly

In [ ]:
FORMAT = pyaudio.paInt16
CHANNELS = 1
RATE = 48000
TARGET_RATE = 16000

CHUNK = 1024
RECORD_SECONDS = 10

DEVICE_INDEX = 0   # replace this with your detected USB mic's index
WAVE_OUTPUT_FILENAME = "output.wav"

In [ ]:
audio = pyaudio.PyAudio()

stream = audio.open(
    format=FORMAT,
    channels=CHANNELS,
    rate=RATE,
    input=True,
    input_device_index=DEVICE_INDEX,
    frames_per_buffer=CHUNK
)

In [ ]:
print("Recording...")

frames = []

for i in range(0, int(RATE / CHUNK * RECORD_SECONDS)):
    data = stream.read(CHUNK, exception_on_overflow=False)
    frames.append(data)

print("Finished recording.")

stream.stop_stream()
stream.close()
audio.terminate()

audio_data = np.frombuffer(b''.join(frames), dtype=np.int16)

# Calcula os fatores para a reamostragem (de 48k para 16k é 1/3)
resampled_data = resample_poly(audio_data, TARGET_RATE, RATE)

# Converte de volta para int16
resampled_data = resampled_data.astype(np.int16)

wf = wave.open(WAVE_OUTPUT_FILENAME, 'wb')

# Define os parâmetros do ÁUDIO REAMOSTRADO
wf.setnchannels(CHANNELS)
wf.setsampwidth(audio.get_sample_size(FORMAT))
wf.setframerate(TARGET_RATE) # <-- MUDANÇA 1: Usar a taxa de amostragem de destino

# Escreve os dados REAMOSTRADOS no arquivo
# .tobytes() converte o array NumPy de volta para bytes
wf.writeframes(resampled_data.tobytes()) # <-- MUDANÇA 2: Usar os dados reamostrados

wf.close()

In [ ]:
!ls

In [ ]:
import IPython.display as ipd
ipd.Audio("output.wav")